In [12]:
import tensorflow as tf
import tf_keras as keras # Keras 2 — required by tensorflow_model_optimization
import tensorflow_model_optimization as tfmot
from tensorflow.keras.datasets import mnist

# Load dataset
(x_train, y_train), (x_test, y_test) = mnist.load_data()
x_train, x_test = x_train / 255.0, x_test / 255.0

# Build a simple model
model = keras.models.Sequential([
    keras.layers.Flatten(input_shape=(28, 28)),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(10, activation='softmax')
])

# Compile the model
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Train the model
model.fit(x_train, y_train, epochs=5, validation_data=(x_test, y_test))

# Apply pruning to the model
pruning_params = {
    'pruning_schedule': tfmot.sparsity.keras.PolynomialDecay(initial_sparsity=0.0, final_sparsity=0.5, begin_step=0, end_step=1000)
}
pruned_model = tfmot.sparsity.keras.prune_low_magnitude(model, **pruning_params)

# Compile the pruned model
pruned_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Train the pruned model to finalize pruning
callbacks = [tfmot.sparsity.keras.UpdatePruningStep()]
pruned_model.fit(x_train, y_train, epochs=2, validation_data=(x_test, y_test), callbacks=callbacks)

# Strip pruning wrappers to remove pruning-specific layers and metadata
pruned_model = tfmot.sparsity.keras.strip_pruning(pruned_model)

# Convert the pruned model to a TensorFlow Lite quantized model
converter = tf.lite.TFLiteConverter.from_keras_model(pruned_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
quantized_model = converter.convert()

Epoch 1/5
1875/1875 [==============================] - 1s 523us/step - loss: 0.2989 - accuracy: 0.9137 - val_loss: 0.1474 - val_accuracy: 0.9574
Epoch 2/5
1875/1875 [==============================] - 1s 455us/step - loss: 0.1462 - accuracy: 0.9560 - val_loss: 0.1050 - val_accuracy: 0.9679
Epoch 3/5
1875/1875 [==============================] - 1s 452us/step - loss: 0.1121 - accuracy: 0.9660 - val_loss: 0.0982 - val_accuracy: 0.9721
Epoch 4/5
1875/1875 [==============================] - 1s 457us/step - loss: 0.0914 - accuracy: 0.9727 - val_loss: 0.0940 - val_accuracy: 0.9700
Epoch 5/5
1875/1875 [==============================] - 1s 439us/step - loss: 0.0782 - accuracy: 0.9758 - val_loss: 0.0775 - val_accuracy: 0.9767
Epoch 1/2
1875/1875 [==============================] - 1s 574us/step - loss: 0.0716 - accuracy: 0.9776 - val_loss: 0.0708 - val_accuracy: 0.9788
Epoch 2/2
1875/1875 [==============================] - 1s 493us/step - loss: 0.0590 - accuracy: 0.9811 - val_loss: 0.0671 - val_ac

INFO:tensorflow:Assets written to: /var/folders/lr/1_ht819n4cl18qfz8jvz6fj80000gp/T/tmp21alcixf/assets
W0000 00:00:1782460684.933181 42164801 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1782460684.933191 42164801 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1782460684.933370 42164801 reader.cc:83] Reading SavedModel from: /var/folders/lr/1_ht819n4cl18qfz8jvz6fj80000gp/T/tmp21alcixf
I0000 00:00:1782460684.933669 42164801 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1782460684.933672 42164801 reader.cc:147] Reading SavedModel debug info (if present) from: /var/folders/lr/1_ht819n4cl18qfz8jvz6fj80000gp/T/tmp21alcixf
I0000 00:00:1782460684.934916 42164801 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
I0000 00:00:1782460684.935079 42164801 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1782460684.941205 42164801 loader.cc:220] Running initialization op on SavedModel bundle

In [13]:
# Measure accuracy of the quantized model using the test set
interpreter = tf.lite.Interpreter(model_content=quantized_model)
interpreter.allocate_tensors()

input_index = interpreter.get_input_details()[0]['index']
output_index = interpreter.get_output_details()[0]['index']

# Evaluate accuracy and precision
correct_predictions = 0
y_pred = []
for i in range(len(x_test)):
    input_data = x_test[i:i+1].astype('float32')
    interpreter.set_tensor(input_index, input_data)
    interpreter.invoke()
    output = interpreter.get_tensor(output_index)
    predicted_label = output.argmax()
    y_pred.append(predicted_label)
    if predicted_label == y_test[i]:
        correct_predictions += 1

accuracy = correct_predictions / len(x_test)
precision = precision_score(y_test, y_pred, average='weighted')
print(f'Quantized model accuracy: {accuracy * 100:.2f}%')
print(f'Quantized model precision: {precision * 100:.2f}%')

Quantized model accuracy: 97.88%
Quantized model precision: 97.88%


/Users/peter.horstedt/git/AI-and-Machine-Learning/.venv/lib/python3.12/site-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


In [14]:
# Measure response time for multiple iterations
start_time = time.time()
for _ in range(25):
    input_data = x_test[0:1].astype('float32')  # Using a single sample to measure response time
    interpreter.set_tensor(input_index, input_data)
    interpreter.invoke()
    _ = interpreter.get_tensor(output_index)
end_time = time.time()

average_response_time = (end_time - start_time) / 25
print(f"Average Response Time: {average_response_time:.4f} seconds")

Average Response Time: 0.0000 seconds


In [16]:
import psutil

# Monitor resource usage
cpu_usage = psutil.cpu_percent()
memory_usage = psutil.virtual_memory().percent

print(f"CPU Usage: {cpu_usage}%")
print(f"Memory Usage: {memory_usage}%")

CPU Usage: 25.6%
Memory Usage: 78.2%


In [17]:
import numpy as np

# Generate large input data for stress testing
large_input = np.tile(x_test, (100, 1, 1))

# Measure performance under stress
start_time = time.time()
for i in range(len(large_input)):
    input_data = large_input[i:i+1].astype('float32')
    interpreter.set_tensor(input_index, input_data)
    interpreter.invoke()
    _ = interpreter.get_tensor(output_index)
end_time = time.time()

stress_response_time = end_time - start_time
print(f"Response Time under Stress: {stress_response_time:.4f} seconds")

Response Time under Stress: 6.2458 seconds
